# Notebook 12 — Reviewer-Driven Reanalysis (v2, corrected)

## Grain AI–MDPI

This notebook performs the confirmatory reanalysis requested in the peer-review report. It preserves the frozen train/calibration/test partitions and does not use the frozen test set to select representations, hyperparameters, calibration rules, or inferential families.

### Corrections in version 2

- Robust Google Drive mounting: it reuses an existing mount or creates a new empty mount point when `/content/drive` already contains files.
- Persistent model-selection checkpoints in Google Drive.
- Explicit preflight state checks, preventing undefined variables such as `roots`.
- Automatic CSV fallback when Parquet support is unavailable.
- Simultaneous discovery/upload of all required ZIP files.
- Exact record-level alignment across representations before paired inference.
- Canonical deterministic APS sensitivity using the cumulative score including the candidate class.
- Clear separation between `FULL`, `FAST`, and `SMOKE` execution modes.

### Execution

For manuscript results, keep:

```python
RUN_MODE = "FULL"
```

Then use **Runtime → Run all**. Do not run cells out of order.

Expected CPU time:

- Colab Pro CPU: approximately 1.5–3.5 hours under favorable allocation.
- Standard Colab CPU: approximately 3–6 hours.
- `FAST`: pipeline verification only; not for the manuscript.
- `SMOKE`: developer self-test on a small deterministic subset; never report its results.

In [ ]:
# ============================================================
# 1. Dependencies, configuration, helpers, and local folders
# ============================================================
import importlib
import importlib.metadata
import os
import subprocess
import sys

# FULL is the only mode intended for manuscript results.
RUN_MODE = os.environ.get("NB12_RUN_MODE", "FULL").strip().upper()
if RUN_MODE not in {"FULL", "FAST", "SMOKE"}:
    raise ValueError("RUN_MODE must be FULL, FAST, or SMOKE.")

required_modules = {
    "joblib": "joblib",
    "statsmodels": "statsmodels",
    "tabulate": "tabulate",
}
missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import shutil
import time
import uuid
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

SEEDS = {
    "cv": 2026081201,
    "final": 2026081202,
    "bootstrap": 2026081203,
    "permutation": 2026081204,
    "conformal": 2026081205,
    "naive": 2026081206,
    "smoke_subset": 2026081207,
}

if RUN_MODE == "FULL":
    N_BOOTSTRAP = 2000
    N_PERM = 200
    N_NAIVE = 5
elif RUN_MODE == "FAST":
    N_BOOTSTRAP = 400
    N_PERM = 40
    N_NAIVE = 2
else:
    N_BOOTSTRAP = 30
    N_PERM = 5
    N_NAIVE = 1

CONFIDENCE = 0.95
EQUIV_MARGIN = 0.01
N_BINS = 15
TARGET_SELECTIVE_COVERAGE = 0.90
NOMINAL_COVERAGES = [0.90, 0.95]
DATASETS = ["dry_bean", "iniap_rice"]

RBF_GRID = [
    {"candidate_id": "rbf_svm_C01", "C": 1.0, "gamma": "scale"},
    {"candidate_id": "rbf_svm_C02", "C": 10.0, "gamma": "scale"},
    {"candidate_id": "rbf_svm_C03", "C": 1.0, "gamma": 0.1},
    {"candidate_id": "rbf_svm_C04", "C": 10.0, "gamma": 0.1},
]
ACTIVE_RBF_GRID = RBF_GRID if RUN_MODE != "SMOKE" else RBF_GRID[:1]

DRY_COMPLETE_INVARIANT = [
    "AspectRation",
    "Eccentricity",
    "Extent",
    "Solidity",
    "roundness",
    "Compactness",
    "ShapeFactor3",
    "ShapeFactor4",
]

NATIVE_BLOCKS = {
    "dry_bean": {
        "scale_dependent": [
            "Area",
            "Perimeter",
            "MajorAxisLength",
            "MinorAxisLength",
            "ConvexArea",
            "EquivDiameter",
            "ShapeFactor1",
            "ShapeFactor2",
        ],
        "dimensionless_shape": DRY_COMPLETE_INVARIANT,
    },
    "iniap_rice": {
        "scale_dependent": [
            "AREA",
            "PERIMETER",
            "MAJOR_AXIS",
            "MINOR_AXIS",
            "CONVEX_AREA",
        ],
        "dimensionless_shape": ["ECCENTRICITY", "EXTENT"],
    },
}

SCALE_EXPONENTS = {
    "dry_bean": {
        "Area": 2,
        "Perimeter": 1,
        "MajorAxisLength": 1,
        "MinorAxisLength": 1,
        "AspectRation": 0,
        "Eccentricity": 0,
        "ConvexArea": 2,
        "EquivDiameter": 1,
        "Extent": 0,
        "Solidity": 0,
        "roundness": 0,
        "Compactness": 0,
        "ShapeFactor1": -1,
        "ShapeFactor2": -1,
        "ShapeFactor3": 0,
        "ShapeFactor4": 0,
    },
    "iniap_rice": {
        "AREA": 2,
        "PERIMETER": 1,
        "MAJOR_AXIS": 1,
        "MINOR_AXIS": 1,
        "ECCENTRICITY": 0,
        "CONVEX_AREA": 2,
        "EXTENT": 0,
    },
}

SELECTIVE_SCORE = {
    "dry_bean": "max_probability",
    "iniap_rice": "top2_margin",
}
SCALE_LEVELS = [-0.05, -0.02, -0.01, 0.0, 0.01, 0.02, 0.05]

WORK_ROOT = Path(
    os.environ.get(
        "NB12_WORK_ROOT",
        "/content/Grain_AI_MDPI_notebook12_v2",
    )
)
INPUT = WORK_ROOT / "00_input"
EXTRACT = WORK_ROOT / "01_extracted"
LOCAL_CHECKPOINT = WORK_ROOT / "02_checkpoints"
OUTROOT = WORK_ROOT / "03_outputs"
PROTOCOL = OUTROOT / "03_protocol"
MODELS = OUTROOT / "05_models"
PREDS = OUTROOT / "06_predictions"
RESULTS = OUTROOT / "07_results" / "12_REVIEWER_REANALYSIS"
FIGURES = OUTROOT / "08_figures" / "12_REVIEWER_REANALYSIS"
TABLES = OUTROOT / "09_tables" / "12_REVIEWER_REANALYSIS"

for folder in [
    INPUT,
    EXTRACT,
    LOCAL_CHECKPOINT,
    PROTOCOL,
    MODELS,
    PREDS,
    RESULTS,
    FIGURES,
    TABLES,
]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT = LOCAL_CHECKPOINT


def require_state(name):
    if not globals().get(name, False):
        raise RuntimeError(
            f"Required state '{name}' is not ready. "
            "Restart the runtime and use Runtime > Run all."
        )


def bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.isin({"true", "1", "yes", "y", "passed"})


def read_frame(parquet_path):
    """Prefer Parquet; automatically use the sibling CSV when needed."""
    parquet_path = Path(parquet_path)
    csv_path = parquet_path.with_suffix(".csv")
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except (ImportError, ModuleNotFoundError, ValueError, OSError) as exc:
            if not csv_path.exists():
                raise RuntimeError(
                    f"Cannot read {parquet_path} and no CSV fallback exists."
                ) from exc
            print(f"Parquet unavailable; using CSV fallback: {csv_path.name}")
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(
        f"Neither {parquet_path} nor {csv_path} exists."
    )


def write_frame(frame, parquet_path):
    """Write Parquet when supported; otherwise write CSV."""
    parquet_path = Path(parquet_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        frame.to_parquet(parquet_path, index=False)
        return parquet_path
    except (ImportError, ModuleNotFoundError, ValueError, OSError):
        csv_path = parquet_path.with_suffix(".csv")
        frame.to_csv(csv_path, index=False, encoding="utf-8-sig")
        return csv_path

print(
    "Mode:", RUN_MODE,
    "| bootstrap:", N_BOOTSTRAP,
    "| permutations:", N_PERM,
    "| naive splits:", N_NAIVE,
    "| grid candidates:", len(ACTIVE_RBF_GRID),
)
print(
    "Python", sys.version.split()[0],
    "| scikit-learn", importlib.metadata.version("scikit-learn"),
    "| NumPy", np.__version__,
    "| pandas", pd.__version__,
    "| CPUs", os.cpu_count(),
)

In [ ]:
# ============================================================
# 2. Robust Drive mount, ZIP discovery, extraction, and checks
# ============================================================
SOURCE_READY = False
roots = {}
zip_paths = {}


def resolve_colab_drive():
    """Reuse an existing mount or create a new empty mount point."""
    try:
        from google.colab import drive, files
    except ModuleNotFoundError:
        return False, None, None

    existing_candidates = [
        Path("/content/drive"),
        Path("/content/drive_nb12"),
    ]
    for mount_point in existing_candidates:
        if (mount_point / "MyDrive").is_dir():
            print("Reusing mounted Google Drive:", mount_point)
            return True, mount_point, files

    standard = Path("/content/drive")
    if not standard.exists() or not any(standard.iterdir()):
        standard.mkdir(parents=True, exist_ok=True)
        try:
            drive.mount(str(standard), force_remount=False)
            if (standard / "MyDrive").is_dir():
                return True, standard, files
        except ValueError as exc:
            print("Standard mount point unavailable:", exc)

    base = Path("/content/drive_nb12")
    if (base / "MyDrive").is_dir():
        return True, base, files
    if base.exists() and any(base.iterdir()):
        base = Path(f"/content/drive_nb12_{uuid.uuid4().hex[:8]}")
    base.mkdir(parents=True, exist_ok=True)
    drive.mount(str(base), force_remount=False)
    if not (base / "MyDrive").is_dir():
        raise RuntimeError("Google Drive mounted, but MyDrive was not found.")
    return True, base, files


IN_COLAB, DRIVE_MOUNT, colab_files = resolve_colab_drive()

if IN_COLAB:
    DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "Grain_Robustness_Q1"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    CHECKPOINT = DRIVE_ROOT / "NOTEBOOK12_CHECKPOINTS_V2"
    CHECKPOINT.mkdir(parents=True, exist_ok=True)
    print("Project folder:", DRIVE_ROOT)
    print("Persistent checkpoints:", CHECKPOINT)
else:
    # Local fallback is included for validation and reproducibility.
    local_candidates = [
        Path(os.environ.get("NB12_SOURCE_ROOT", "/mnt/data")),
        Path.cwd(),
        Path("/content"),
    ]
    DRIVE_ROOT = next(
        (candidate for candidate in local_candidates if candidate.exists()),
        Path.cwd(),
    )
    CHECKPOINT = LOCAL_CHECKPOINT
    colab_files = None
    print("Google Colab was not detected; using local source root:", DRIVE_ROOT)

REQUIRED = {
    "notebook02": "NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip",
    "notebook03": "NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip",
    "notebook04": "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip",
    "notebook05": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
    "notebook06": "NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS.zip",
    "notebook08": "NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS.zip",
}
OPTIONAL = {
    "notebook07": "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip",
    "notebook09": "NOTEBOOK09_ABLATION_INTERPRETABILITY_OUTPUTS.zip",
    "notebook10": "NOTEBOOK10_MEASUREMENT_UNCERTAINTY_OUTPUTS.zip",
    "notebook11": "NOTEBOOK11_MANUSCRIPT_TABLES_FIGURES_OUTPUTS_V4.zip",
}


def newest(filename):
    # Check exact locations first; this avoids slow recursive scans.
    exact_candidates = [
        DRIVE_ROOT / filename,
        INPUT / filename,
        Path("/content") / filename,
        Path("/mnt/data") / filename,
        Path.cwd() / filename,
    ]
    valid = [path for path in exact_candidates if path.is_file()]

    # In Drive, users may have placed the ZIP in a project subfolder.
    # Restrict recursive search to the dedicated project root only.
    if not valid and DRIVE_ROOT.exists():
        try:
            valid.extend(path for path in DRIVE_ROOT.rglob(filename) if path.is_file())
        except (OSError, PermissionError):
            pass

    return max(valid, key=lambda path: path.stat().st_mtime) if valid else None


missing_required = [
    filename for filename in REQUIRED.values() if newest(filename) is None
]
if missing_required and IN_COLAB:
    print("Upload all missing required ZIP files in one selection:")
    for filename in missing_required:
        print(" -", filename)
    uploaded = colab_files.upload()
    for filename in missing_required:
        if filename not in uploaded:
            raise FileNotFoundError(
                f"Required ZIP was not uploaded: {filename}"
            )
        destination = INPUT / filename
        destination.write_bytes(uploaded[filename])

missing_after_upload = [
    filename for filename in REQUIRED.values() if newest(filename) is None
]
if missing_after_upload:
    raise FileNotFoundError(
        "Required ZIP files are missing: " + ", ".join(missing_after_upload)
    )

for key, filename in REQUIRED.items():
    zip_paths[key] = newest(filename)
for key, filename in OPTIONAL.items():
    zip_paths[key] = newest(filename)
    if zip_paths[key] is None:
        print("Optional ZIP not found:", filename)


def safe_extract(source, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(source, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise zipfile.BadZipFile(
                f"CRC validation failed in {source.name}: {bad_member}"
            )
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise RuntimeError(
                    f"Unsafe path inside ZIP {source.name}: {member.filename}"
                )
        archive.extractall(destination)


for key, source in zip_paths.items():
    if source is None:
        continue
    destination = EXTRACT / key
    safe_extract(source, destination)
    roots[key] = destination
    print("Extracted:", source.name)


def find_one(root, name):
    matches = sorted(root.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} was not found under {root}")
    return matches[0]


rep_catalog = pd.read_csv(
    find_one(roots["notebook02"], "representation_catalog.csv")
)
old_selection = pd.read_csv(
    find_one(roots["notebook04"], "03_selected_model_configurations.csv")
)
val06 = pd.read_csv(
    find_one(roots["notebook06"], "02_final_inference_validation.csv")
)
val08 = pd.read_csv(
    find_one(roots["notebook08"], "05_final_conformal_validation_v2.csv")
)

if not bool_series(val06["passed"]).all():
    display(val06)
    raise RuntimeError("Notebook 06 source validation failed.")
if not bool_series(val08["passed"]).all():
    display(val08)
    raise RuntimeError("Notebook 08 v2 source validation failed.")

source_inventory = pd.DataFrame(
    [
        {
            "source": key,
            "filename": path.name if path else "",
            "path": str(path) if path else "",
            "required": key in REQUIRED,
            "found": path is not None,
        }
        for key, path in zip_paths.items()
    ]
)
source_inventory.to_csv(
    RESULTS / "00_source_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)
SOURCE_READY = True
print("Source preflight: PASSED")
display(source_inventory)

In [ ]:
# ============================================================
# 3. Representation audit and corrected frozen data frames
# ============================================================
require_state("SOURCE_READY")
REPRESENTATIONS_READY = False

META = [
    "dataset_id",
    "record_id",
    "source_excel_row",
    "feature_fingerprint_sha256",
    "full_row_fingerprint_sha256",
    "duplicate_group_id",
    "exact_duplicate_count",
    "class_label",
    "fold_id",
    "partition",
]


def load_split(dataset, representation):
    path = (
        roots["notebook03"]
        / "05_frozen_splits"
        / dataset
        / f"{representation}_with_split.parquet"
    )
    frame = read_frame(path)
    required_columns = {
        "record_id",
        "duplicate_group_id",
        "class_label",
        "fold_id",
        "partition",
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise RuntimeError(
            f"Missing columns in {dataset}/{representation}: {missing_columns}"
        )
    for column in ["record_id", "duplicate_group_id", "class_label"]:
        frame[column] = frame[column].astype(str)
    return frame


base_representation_names = [
    "R1_full_native",
    "R2_measured_core",
    "R3_invariant_core",
    "R4_hybrid_log_area",
]
old = {
    (dataset, representation): load_split(dataset, representation)
    for dataset in DATASETS
    for representation in base_representation_names
}

# Developer-only small deterministic subset. It is never used in FULL/FAST.
if RUN_MODE == "SMOKE":
    rng = np.random.default_rng(SEEDS["smoke_subset"])
    for dataset in DATASETS:
        reference = old[(dataset, "R1_full_native")]
        group_table = (
            reference.groupby("duplicate_group_id", as_index=False, sort=True)
            .first()[
                [
                    "duplicate_group_id",
                    "partition",
                    "fold_id",
                    "class_label",
                ]
            ]
        )
        selected_groups = []
        for _, stratum in group_table.groupby(
            ["partition", "fold_id", "class_label"], dropna=False
        ):
            n_take = min(25, len(stratum))
            positions = rng.choice(len(stratum), size=n_take, replace=False)
            selected_groups.extend(
                stratum.iloc[positions]["duplicate_group_id"].astype(str).tolist()
            )
        selected_groups = set(selected_groups)
        for representation in base_representation_names:
            old[(dataset, representation)] = old[(dataset, representation)].loc[
                old[(dataset, representation)]["duplicate_group_id"].isin(selected_groups)
            ].copy()
        print(dataset, "SMOKE rows:", len(old[(dataset, "R1_full_native")]))

frames = {}
features = {}
origins = {}

for dataset in DATASETS:
    for representation in ["R1_full_native", "R2_measured_core"]:
        frames[(dataset, representation)] = old[(dataset, representation)].copy()
        features[(dataset, representation)] = [
            column
            for column in frames[(dataset, representation)].columns
            if column not in META
        ]
        origins[(dataset, representation)] = "existing_frozen"

    frames[(dataset, "R3_reduced_current")] = old[
        (dataset, "R3_invariant_core")
    ].copy()
    features[(dataset, "R3_reduced_current")] = [
        column
        for column in frames[(dataset, "R3_reduced_current")].columns
        if column not in META
    ]
    origins[(dataset, "R3_reduced_current")] = (
        "original_reduced_invariant_core"
    )

    frames[(dataset, "R4_reduced_log_area_current")] = old[
        (dataset, "R4_hybrid_log_area")
    ].copy()
    features[(dataset, "R4_reduced_log_area_current")] = [
        column
        for column in frames[(dataset, "R4_reduced_log_area_current")].columns
        if column not in META
    ]
    origins[(dataset, "R4_reduced_log_area_current")] = (
        "original_reduced_invariant_plus_log_area"
    )

# Dry Bean: all eight native dimensionless descriptors requested by the reviewer.
dry = old[("dry_bean", "R1_full_native")].copy()
missing = sorted(set(DRY_COMPLETE_INVARIANT) - set(dry.columns))
if missing:
    raise RuntimeError(f"Missing Dry Bean invariant variables: {missing}")

frames[("dry_bean", "R3_complete_invariant")] = dry[
    META + DRY_COMPLETE_INVARIANT
].copy()
features[("dry_bean", "R3_complete_invariant")] = DRY_COMPLETE_INVARIANT.copy()
origins[("dry_bean", "R3_complete_invariant")] = (
    "all_eight_dimensionless_R1_predictors"
)

r4_complete = frames[("dry_bean", "R3_complete_invariant")].copy()
if (dry["Area"] <= 0).any():
    raise RuntimeError("Dry Bean Area contains non-positive values.")
r4_complete["log_area"] = np.log(dry["Area"].to_numpy(dtype=float))
frames[("dry_bean", "R4_complete_log_area")] = r4_complete
features[("dry_bean", "R4_complete_log_area")] = (
    DRY_COMPLETE_INVARIANT + ["log_area"]
)
origins[("dry_bean", "R4_complete_log_area")] = (
    "complete_invariant_plus_log_pixel_area"
)

# Rice: the supplied engineered invariant set is the complete available set.
frames[("iniap_rice", "R3_complete_invariant")] = old[
    ("iniap_rice", "R3_invariant_core")
].copy()
features[("iniap_rice", "R3_complete_invariant")] = [
    column
    for column in frames[("iniap_rice", "R3_complete_invariant")].columns
    if column not in META
]
origins[("iniap_rice", "R3_complete_invariant")] = (
    "complete_engineered_invariant_set"
)
frames[("iniap_rice", "R4_complete_log_area")] = old[
    ("iniap_rice", "R4_hybrid_log_area")
].copy()
features[("iniap_rice", "R4_complete_log_area")] = [
    column
    for column in frames[("iniap_rice", "R4_complete_log_area")].columns
    if column not in META
]
origins[("iniap_rice", "R4_complete_log_area")] = (
    "complete_invariant_plus_log_pixel_area"
)

REPS = {
    "dry_bean": [
        "R1_full_native",
        "R2_measured_core",
        "R3_reduced_current",
        "R4_reduced_log_area_current",
        "R3_complete_invariant",
        "R4_complete_log_area",
    ],
    "iniap_rice": [
        "R1_full_native",
        "R2_measured_core",
        "R3_complete_invariant",
        "R4_complete_log_area",
    ],
}

# Confirm that every representation contains the same records and frozen partitions.
for dataset in DATASETS:
    reference = frames[(dataset, "R1_full_native")][
        ["record_id", "duplicate_group_id", "class_label", "fold_id", "partition"]
    ].sort_values("record_id").reset_index(drop=True)
    for representation in REPS[dataset]:
        candidate = frames[(dataset, representation)][
            ["record_id", "duplicate_group_id", "class_label", "fold_id", "partition"]
        ].sort_values("record_id").reset_index(drop=True)
        if not candidate.equals(reference):
            raise RuntimeError(
                f"Frozen metadata mismatch: {dataset}/{representation}"
            )

rows = []
for dataset in DATASETS:
    for representation in REPS[dataset]:
        frame = frames[(dataset, representation)]
        feature_list = features[(dataset, representation)]
        rows.append(
            {
                "dataset_id": dataset,
                "representation": representation,
                "origin": origins[(dataset, representation)],
                "n_features": len(feature_list),
                "features": " | ".join(feature_list),
                "rows": len(frame),
                "missing_values": int(frame[feature_list].isna().sum().sum()),
                "partitions": " | ".join(sorted(frame["partition"].unique())),
                "run_mode": RUN_MODE,
            }
        )

representation_audit = pd.DataFrame(rows)
representation_audit.to_csv(
    TABLES / "Table_01_representation_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
REPRESENTATIONS_READY = True
print("Representation audit: PASSED")
display(representation_audit)

In [ ]:
# ============================================================
# 4. RBF-SVM functions, training-only selection, and final fit
# ============================================================
require_state("REPRESENTATIONS_READY")
MODEL_STAGE_READY = False


def normalize_gamma(value):
    if isinstance(value, str):
        value = value.strip()
        return value if value in {"scale", "auto"} else float(value)
    return float(value)


def build_model(parameters, seed):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                SVC(
                    C=float(parameters["C"]),
                    gamma=normalize_gamma(parameters["gamma"]),
                    kernel="rbf",
                    probability=True,
                    random_state=seed,
                    cache_size=3000,
                ),
            ),
        ]
    )


def aligned_proba(model, X, class_order):
    raw = np.asarray(model.predict_proba(np.asarray(X, dtype=float)), dtype=float)
    model_classes = np.asarray(model.classes_, dtype=str)
    class_order = np.asarray(class_order, dtype=str)
    if set(model_classes) != set(class_order):
        raise RuntimeError("Class order mismatch.")
    output = np.zeros((len(raw), len(class_order)), dtype=float)
    destination = {label: index for index, label in enumerate(class_order)}
    for source_index, label in enumerate(model_classes):
        output[:, destination[label]] = raw[:, source_index]
    row_sums = output.sum(axis=1, keepdims=True)
    if not np.isfinite(output).all() or (row_sums <= 0).any():
        raise RuntimeError("Invalid predicted probabilities.")
    return output / row_sums


def labels_from_proba(probabilities, class_order):
    return np.asarray(class_order, dtype=str)[
        np.asarray(probabilities, dtype=float).argmax(axis=1)
    ]


def multiclass_brier(y_true, probabilities, class_order):
    y_true = np.asarray(y_true, dtype=str)
    probabilities = np.asarray(probabilities, dtype=float)
    class_order = np.asarray(class_order, dtype=str)
    indicator = np.zeros_like(probabilities)
    position = {label: index for index, label in enumerate(class_order)}
    for row_index, label in enumerate(y_true):
        indicator[row_index, position[label]] = 1.0
    return float(np.mean(np.sum((probabilities - indicator) ** 2, axis=1)))


def metrics(y_true, probabilities, class_order):
    predictions = labels_from_proba(probabilities, class_order)
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, predictions)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                predictions,
                labels=class_order,
                average="macro",
                zero_division=0,
            )
        ),
        "mcc": float(matthews_corrcoef(y_true, predictions)),
        "log_loss": float(log_loss(y_true, probabilities, labels=class_order)),
        "brier_multiclass": multiclass_brier(
            y_true, probabilities, class_order
        ),
    }


def scale_temperature(probabilities, temperature):
    return softmax(
        np.log(np.clip(np.asarray(probabilities, dtype=float), 1e-15, 1.0))
        / float(temperature),
        axis=1,
    )


def fit_temperature(probabilities, y_true, class_order):
    def objective(log_temperature):
        return log_loss(
            y_true,
            scale_temperature(probabilities, np.exp(log_temperature)),
            labels=class_order,
        )

    result = minimize_scalar(
        objective,
        bounds=(-3.0, 3.0),
        method="bounded",
        options={"xatol": 1e-8},
    )
    if not result.success:
        raise RuntimeError("Temperature optimization failed.")
    return float(np.exp(result.x))


selection_signature = {
    "protocol": "notebook12_v2_corrected",
    "run_mode": RUN_MODE,
    "grid": ACTIVE_RBF_GRID,
    "representations": REPS,
    "seed": SEEDS["cv"],
    "sklearn": importlib.metadata.version("scikit-learn"),
}
selection_hash = hashlib.sha256(
    json.dumps(selection_signature, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
checkpoint = CHECKPOINT / f"selection_{RUN_MODE}_{selection_hash}.csv"

selection_rows = []
completed = set()
if checkpoint.exists():
    previous = pd.read_csv(checkpoint)
    expected_columns = {
        "dataset_id",
        "representation",
        "candidate_id",
        "C",
        "gamma",
        "macro_f1_mean",
        "log_loss_mean",
    }
    if expected_columns.issubset(previous.columns):
        previous = previous.drop_duplicates(
            ["dataset_id", "representation", "candidate_id"], keep="last"
        )
        selection_rows = previous.to_dict("records")
        completed = set(
            zip(
                previous["dataset_id"],
                previous["representation"],
                previous["candidate_id"],
            )
        )
        print("Resuming from", len(previous), "candidate summaries.")
    else:
        backup = checkpoint.with_suffix(f".invalid_{int(time.time())}.csv")
        checkpoint.rename(backup)
        print("Invalid old checkpoint moved to:", backup)

selection_start = time.perf_counter()
for dataset in DATASETS:
    for representation in REPS[dataset]:
        frame = frames[(dataset, representation)]
        feature_list = features[(dataset, representation)]
        train = frame.loc[frame["partition"] == "train"].copy()
        folds = sorted(train["fold_id"].astype(int).unique().tolist())
        if len(folds) < 2:
            raise RuntimeError(
                f"Insufficient training folds for {dataset}/{representation}."
            )
        class_order = np.sort(train["class_label"].astype(str).unique())

        for candidate in ACTIVE_RBF_GRID:
            key = (dataset, representation, candidate["candidate_id"])
            if key in completed:
                continue
            fold_rows = []
            candidate_start = time.perf_counter()
            for validation_fold in folds:
                fit_frame = train.loc[
                    train["fold_id"].astype(int) != validation_fold
                ]
                validation_frame = train.loc[
                    train["fold_id"].astype(int) == validation_fold
                ]
                model = build_model(candidate, SEEDS["cv"])
                model.fit(
                    fit_frame[feature_list].to_numpy(dtype=float),
                    fit_frame["class_label"].astype(str).to_numpy(),
                )
                probabilities = aligned_proba(
                    model,
                    validation_frame[feature_list].to_numpy(dtype=float),
                    class_order,
                )
                fold_rows.append(
                    metrics(
                        validation_frame["class_label"].astype(str).to_numpy(),
                        probabilities,
                        class_order,
                    )
                )
            fold_metrics = pd.DataFrame(fold_rows)
            selection_rows.append(
                {
                    "dataset_id": dataset,
                    "representation": representation,
                    "candidate_id": candidate["candidate_id"],
                    "C": candidate["C"],
                    "gamma": candidate["gamma"],
                    "n_folds": len(folds),
                    "macro_f1_mean": float(fold_metrics["macro_f1"].mean()),
                    "macro_f1_sd": float(fold_metrics["macro_f1"].std(ddof=1)),
                    "log_loss_mean": float(fold_metrics["log_loss"].mean()),
                    "accuracy_mean": float(fold_metrics["accuracy"].mean()),
                    "elapsed_seconds": time.perf_counter() - candidate_start,
                    "run_mode": RUN_MODE,
                    "selection_hash": selection_hash,
                }
            )
            pd.DataFrame(selection_rows).to_csv(
                checkpoint, index=False, encoding="utf-8-sig"
            )
            print(dataset, representation, candidate["candidate_id"], "done")

selection = pd.DataFrame(selection_rows)
expected_candidate_count = sum(len(value) for value in REPS.values()) * len(
    ACTIVE_RBF_GRID
)
if len(selection) != expected_candidate_count:
    raise RuntimeError(
        f"Expected {expected_candidate_count} candidate summaries; got {len(selection)}."
    )

selected = (
    selection.sort_values(
        [
            "dataset_id",
            "representation",
            "macro_f1_mean",
            "log_loss_mean",
            "candidate_id",
        ],
        ascending=[True, True, False, True, True],
    )
    .groupby(["dataset_id", "representation"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
selection.to_csv(
    TABLES / "Table_02_training_only_rbf_selection_all_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)
selected.to_csv(
    TABLES / "Table_03_training_selected_rbf_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)
print(
    "Selection minutes:",
    round((time.perf_counter() - selection_start) / 60.0, 2),
)
display(selected)

bundles = {}
timing_rows = []
metadata_rows = []
for _, selected_row in selected.iterrows():
    dataset = selected_row["dataset_id"]
    representation = selected_row["representation"]
    frame = frames[(dataset, representation)]
    feature_list = features[(dataset, representation)]
    train = frame.loc[frame["partition"] == "train"].copy()
    calibration = frame.loc[frame["partition"] == "calibration"].copy()
    test = frame.loc[frame["partition"] == "test"].copy()
    class_order = np.sort(train["class_label"].astype(str).unique())
    parameters = {
        "C": float(selected_row["C"]),
        "gamma": normalize_gamma(selected_row["gamma"]),
    }
    model = build_model(parameters, SEEDS["final"])
    start = time.perf_counter()
    model.fit(
        train[feature_list].to_numpy(dtype=float),
        train["class_label"].astype(str).to_numpy(),
    )
    fit_seconds = time.perf_counter() - start

    calibration_native = aligned_proba(
        model,
        calibration[feature_list].to_numpy(dtype=float),
        class_order,
    )
    temperature = fit_temperature(
        calibration_native,
        calibration["class_label"].astype(str).to_numpy(),
        class_order,
    )
    calibration_scaled = scale_temperature(calibration_native, temperature)

    start = time.perf_counter()
    test_native = aligned_proba(
        model, test[feature_list].to_numpy(dtype=float), class_order
    )
    inference_seconds = time.perf_counter() - start
    test_scaled = scale_temperature(test_native, temperature)

    bundles[(dataset, representation)] = {
        "model": model,
        "features": feature_list,
        "class_order": class_order,
        "temperature": temperature,
        "train": train.reset_index(drop=True),
        "calibration": calibration.reset_index(drop=True),
        "test": test.reset_index(drop=True),
        "cal_native": calibration_native,
        "cal_scaled": calibration_scaled,
        "test_native": test_native,
        "test_scaled": test_scaled,
    }

    model_dir = MODELS / dataset / representation
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "rbf_svm_reanalysis.joblib"
    joblib.dump(model, model_path)

    for partition_name, partition_frame, native, scaled in [
        ("calibration", calibration.reset_index(drop=True), calibration_native, calibration_scaled),
        ("test", test.reset_index(drop=True), test_native, test_scaled),
    ]:
        output = partition_frame[
            [
                "dataset_id",
                "record_id",
                "duplicate_group_id",
                "class_label",
                "partition",
            ]
        ].copy()
        output["predicted_native"] = labels_from_proba(native, class_order)
        output["predicted_temperature_scaled"] = labels_from_proba(
            scaled, class_order
        )
        for class_index, class_label in enumerate(class_order):
            output[f"class_{class_index}"] = class_label
            output[f"prob_native_{class_index}"] = native[:, class_index]
            output[f"prob_temperature_scaled_{class_index}"] = scaled[:, class_index]
        write_frame(
            output,
            PREDS
            / dataset
            / representation
            / f"rbf_svm_{partition_name}_predictions.parquet",
        )

    timing_rows.append(
        {
            "dataset_id": dataset,
            "representation": representation,
            "fit_seconds": fit_seconds,
            "test_inference_seconds": inference_seconds,
            "test_rows": len(test),
            "milliseconds_per_test_row": 1000.0
            * inference_seconds
            / max(len(test), 1),
        }
    )
    metadata_rows.append(
        {
            "dataset_id": dataset,
            "representation": representation,
            "candidate_id": selected_row["candidate_id"],
            "C": parameters["C"],
            "gamma": parameters["gamma"],
            "temperature": temperature,
            "features": " | ".join(feature_list),
            "class_order": " | ".join(class_order),
            "model_path": str(model_path.relative_to(OUTROOT)),
        }
    )

fit_timing = pd.DataFrame(timing_rows)
model_metadata = pd.DataFrame(metadata_rows)
fit_timing.to_csv(
    TABLES / "Table_04_computational_timing.csv",
    index=False,
    encoding="utf-8-sig",
)
model_metadata.to_csv(
    RESULTS / "01_model_metadata.csv",
    index=False,
    encoding="utf-8-sig",
)
MODEL_STAGE_READY = True
print("Model stage: PASSED")
display(fit_timing)

In [ ]:
# ============================================================
# 5. Test metrics, group bootstrap, McNemar, and equivalence
# ============================================================
require_state("MODEL_STAGE_READY")
INFERENCE_READY = False

ALL_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "mcc",
    "log_loss",
    "brier_multiclass",
]
CONTRASTS = {
    "dry_bean": [
        ("R1_full_native", "R2_measured_core"),
        ("R1_full_native", "R3_complete_invariant"),
        ("R1_full_native", "R3_reduced_current"),
        ("R1_full_native", "R4_complete_log_area"),
        ("R2_measured_core", "R3_complete_invariant"),
        ("R3_complete_invariant", "R4_complete_log_area"),
    ],
    "iniap_rice": [
        ("R1_full_native", "R2_measured_core"),
        ("R1_full_native", "R3_complete_invariant"),
        ("R1_full_native", "R4_complete_log_area"),
        ("R2_measured_core", "R3_complete_invariant"),
        ("R3_complete_invariant", "R4_complete_log_area"),
    ],
}


def aligned_test(bundle, reference_record_ids):
    test = bundle["test"].copy()
    index = pd.Index(test["record_id"].astype(str))
    positions = index.get_indexer(reference_record_ids)
    if (positions < 0).any() or len(set(positions.tolist())) != len(positions):
        raise RuntimeError("Record alignment failed across representations.")
    aligned_frame = test.iloc[positions].reset_index(drop=True)
    aligned_probabilities = bundle["test_native"][positions]
    if not aligned_frame["record_id"].astype(str).equals(
        pd.Series(reference_record_ids, dtype=str)
    ):
        raise RuntimeError("Aligned record IDs do not match the reference.")
    return aligned_frame, aligned_probabilities


point_rows = []
for (dataset, representation), bundle in bundles.items():
    point_rows.append(
        {
            "dataset_id": dataset,
            "representation": representation,
            **metrics(
                bundle["test"]["class_label"].astype(str).to_numpy(),
                bundle["test_native"],
                bundle["class_order"],
            ),
        }
    )
point = pd.DataFrame(point_rows)
point.to_csv(
    TABLES / "Table_05_corrected_representation_test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)


def percentile(values, level=CONFIDENCE):
    alpha = (1.0 - level) / 2.0
    return (
        float(np.quantile(values, alpha)),
        float(np.quantile(values, 1.0 - alpha)),
    )


def empirical_p(values, null=0.0):
    values = np.asarray(values, dtype=float)
    lower = (np.sum(values <= null) + 1) / (len(values) + 1)
    upper = (np.sum(values >= null) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower, upper)))


bootstrap = {}
reference_ids_by_dataset = {}
rng = np.random.default_rng(SEEDS["bootstrap"])

for dataset in DATASETS:
    reference = bundles[(dataset, "R1_full_native")]["test"].copy()
    reference = reference.sort_values("record_id", kind="mergesort").reset_index(drop=True)
    reference_ids = reference["record_id"].astype(str).tolist()
    reference_ids_by_dataset[dataset] = reference_ids

    groups = reference[["duplicate_group_id", "class_label"]].drop_duplicates()
    if groups.groupby("duplicate_group_id")["class_label"].nunique().max() != 1:
        raise RuntimeError(f"Multiple labels in a test group for {dataset}.")
    group_indices = {
        group_id: reference.index[
            reference["duplicate_group_id"] == group_id
        ].to_numpy(dtype=int)
        for group_id in groups["duplicate_group_id"]
    }
    groups_by_class = {
        class_label: groups.loc[
            groups["class_label"] == class_label, "duplicate_group_id"
        ].to_numpy()
        for class_label in sorted(groups["class_label"].unique())
    }

    bootstrap_indices = []
    for _ in range(N_BOOTSTRAP):
        sampled_indices = []
        for _, class_groups in groups_by_class.items():
            sampled_groups = rng.choice(
                class_groups, size=len(class_groups), replace=True
            )
            for group_id in sampled_groups:
                sampled_indices.extend(group_indices[group_id].tolist())
        bootstrap_indices.append(np.asarray(sampled_indices, dtype=int))

    for representation in REPS[dataset]:
        bundle = bundles[(dataset, representation)]
        test, probabilities = aligned_test(bundle, reference_ids)
        y_true = test["class_label"].astype(str).to_numpy()
        metric_arrays = {
            metric_name: np.empty(N_BOOTSTRAP, dtype=float)
            for metric_name in ALL_METRICS
        }
        for bootstrap_index, sample_indices in enumerate(bootstrap_indices):
            output = metrics(
                y_true[sample_indices],
                probabilities[sample_indices],
                bundle["class_order"],
            )
            for metric_name in ALL_METRICS:
                metric_arrays[metric_name][bootstrap_index] = output[metric_name]
        bootstrap[(dataset, representation)] = metric_arrays

contrast_rows = []
mcnemar_rows = []
equivalence_rows = []

for dataset in DATASETS:
    reference_ids = reference_ids_by_dataset[dataset]
    for representation_a, representation_b in CONTRASTS[dataset]:
        point_a = point.loc[
            (point["dataset_id"] == dataset)
            & (point["representation"] == representation_a)
        ].iloc[0]
        point_b = point.loc[
            (point["dataset_id"] == dataset)
            & (point["representation"] == representation_b)
        ].iloc[0]

        for metric_name in ALL_METRICS:
            delta = (
                bootstrap[(dataset, representation_a)][metric_name]
                - bootstrap[(dataset, representation_b)][metric_name]
            )
            ci_low, ci_high = percentile(delta)
            contrast_rows.append(
                {
                    "dataset_id": dataset,
                    "representation_a": representation_a,
                    "representation_b": representation_b,
                    "contrast": f"{representation_a}_minus_{representation_b}",
                    "metric": metric_name,
                    "delta_a_minus_b": float(
                        point_a[metric_name] - point_b[metric_name]
                    ),
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                    "bootstrap_p_raw": empirical_p(delta),
                    "bootstrap_p_minimum_resolution": 2.0
                    / (N_BOOTSTRAP + 1),
                    "n_bootstrap": N_BOOTSTRAP,
                }
            )

        bundle_a = bundles[(dataset, representation_a)]
        bundle_b = bundles[(dataset, representation_b)]
        frame_a, probabilities_a = aligned_test(bundle_a, reference_ids)
        frame_b, probabilities_b = aligned_test(bundle_b, reference_ids)
        if not frame_a["class_label"].equals(frame_b["class_label"]):
            raise RuntimeError("McNemar labels are not aligned.")
        y_true = frame_a["class_label"].astype(str).to_numpy()
        labels_a = labels_from_proba(probabilities_a, bundle_a["class_order"])
        labels_b = labels_from_proba(probabilities_b, bundle_b["class_order"])
        correct_a = labels_a == y_true
        correct_b = labels_b == y_true
        b_count = int(np.sum(correct_a & ~correct_b))
        c_count = int(np.sum(~correct_a & correct_b))
        discordant_total = b_count + c_count
        exact_p = (
            1.0
            if discordant_total == 0
            else float(
                binomtest(
                    min(b_count, c_count),
                    n=discordant_total,
                    p=0.5,
                    alternative="two-sided",
                ).pvalue
            )
        )
        mcnemar_rows.append(
            {
                "dataset_id": dataset,
                "representation_a": representation_a,
                "representation_b": representation_b,
                "b_a_correct_b_wrong": b_count,
                "c_a_wrong_b_correct": c_count,
                "discordant_total": discordant_total,
                "mcnemar_exact_p_raw": exact_p,
            }
        )

        if dataset == "iniap_rice":
            delta = (
                bootstrap[(dataset, representation_a)]["macro_f1"]
                - bootstrap[(dataset, representation_b)]["macro_f1"]
            )
            ci_90_low, ci_90_high = percentile(delta, 0.90)
            p_lower = (np.sum(delta <= -EQUIV_MARGIN) + 1) / (
                len(delta) + 1
            )
            p_upper = (np.sum(delta >= EQUIV_MARGIN) + 1) / (
                len(delta) + 1
            )
            equivalence_rows.append(
                {
                    "dataset_id": dataset,
                    "representation_a": representation_a,
                    "representation_b": representation_b,
                    "margin": EQUIV_MARGIN,
                    "delta_macro_f1": float(
                        point_a["macro_f1"] - point_b["macro_f1"]
                    ),
                    "bootstrap_90_ci_low": ci_90_low,
                    "bootstrap_90_ci_high": ci_90_high,
                    "empirical_tost_p_lower": float(p_lower),
                    "empirical_tost_p_upper": float(p_upper),
                    "equivalent_at_0_05": bool(
                        ci_90_low > -EQUIV_MARGIN
                        and ci_90_high < EQUIV_MARGIN
                        and p_lower < 0.05
                        and p_upper < 0.05
                    ),
                }
            )

contrasts = pd.DataFrame(contrast_rows)
mcnemar = pd.DataFrame(mcnemar_rows)
equivalence = pd.DataFrame(equivalence_rows)

contrasts["bootstrap_p_holm"] = np.nan
contrasts["reject_holm_0_05"] = False
for (_, _), indices in contrasts.groupby(["dataset_id", "metric"]).groups.items():
    reject, adjusted, _, _ = multipletests(
        contrasts.loc[indices, "bootstrap_p_raw"].to_numpy(dtype=float),
        alpha=0.05,
        method="holm",
    )
    contrasts.loc[indices, "bootstrap_p_holm"] = adjusted
    contrasts.loc[indices, "reject_holm_0_05"] = reject

mcnemar["mcnemar_exact_p_holm"] = np.nan
mcnemar["reject_holm_0_05"] = False
for _, indices in mcnemar.groupby("dataset_id").groups.items():
    reject, adjusted, _, _ = multipletests(
        mcnemar.loc[indices, "mcnemar_exact_p_raw"].to_numpy(dtype=float),
        alpha=0.05,
        method="holm",
    )
    mcnemar.loc[indices, "mcnemar_exact_p_holm"] = adjusted
    mcnemar.loc[indices, "reject_holm_0_05"] = reject

contrasts.to_csv(
    TABLES / "Table_06_corrected_paired_representation_contrasts.csv",
    index=False,
    encoding="utf-8-sig",
)
mcnemar.to_csv(
    TABLES / "Table_07_exact_mcnemar_discordant_counts.csv",
    index=False,
    encoding="utf-8-sig",
)
equivalence.to_csv(
    TABLES / "Table_08_rice_bootstrap_TOST_equivalence.csv",
    index=False,
    encoding="utf-8-sig",
)
INFERENCE_READY = True
print("Paired inference: PASSED")
display(contrasts.loc[contrasts["metric"] == "macro_f1"])
display(mcnemar)
display(equivalence)

In [ ]:
# ============================================================
# 6. Controlled R1 block importance and image-space scale bias
# ============================================================
require_state("INFERENCE_READY")
CONTROLLED_ANALYSES_READY = False


def group_frame(frame):
    feature_columns = [column for column in frame.columns if column not in META]
    if (frame.groupby("duplicate_group_id")["class_label"].nunique() > 1).any():
        raise RuntimeError("A duplicate group contains multiple labels.")
    if (
        frame.groupby("duplicate_group_id")[feature_columns]
        .nunique(dropna=False)
        .max(axis=1)
        > 1
    ).any():
        raise RuntimeError("A duplicate group contains multiple feature vectors.")
    return frame.groupby("duplicate_group_id", as_index=False, sort=True).first()


def permuted_macro_f1_drop(
    model,
    X,
    y_true,
    class_order,
    column_indices,
    permutation,
    baseline_macro_f1,
):
    perturbed = X.copy()
    perturbed[:, column_indices] = X[permutation][:, column_indices]
    perturbed_probabilities = aligned_proba(model, perturbed, class_order)
    perturbed_macro_f1 = metrics(
        y_true, perturbed_probabilities, class_order
    )["macro_f1"]
    return float(baseline_macro_f1 - perturbed_macro_f1)


importance_rows = []
scale_rows = []
for dataset_index, dataset in enumerate(DATASETS):
    bundle = bundles[(dataset, "R1_full_native")]
    test = group_frame(bundle["test"])
    feature_list = bundle["features"]
    position = {feature: index for index, feature in enumerate(feature_list)}
    X = test[feature_list].to_numpy(dtype=float)
    y_true = test["class_label"].astype(str).to_numpy()
    baseline_probabilities = aligned_proba(
        bundle["model"], X, bundle["class_order"]
    )
    baseline_metrics = metrics(
        y_true, baseline_probabilities, bundle["class_order"]
    )
    baseline_labels = labels_from_proba(
        baseline_probabilities, bundle["class_order"]
    )
    rng = np.random.default_rng(SEEDS["permutation"] + dataset_index)
    shared_permutations = [rng.permutation(len(X)) for _ in range(N_PERM)]

    for block_name, block_features in NATIVE_BLOCKS[dataset].items():
        missing = sorted(set(block_features) - set(feature_list))
        if missing:
            raise RuntimeError(
                f"Missing block variables for {dataset}: {missing}"
            )
        full_columns = [position[feature] for feature in block_features]
        full_drops = np.asarray(
            [
                permuted_macro_f1_drop(
                    bundle["model"],
                    X,
                    y_true,
                    bundle["class_order"],
                    full_columns,
                    permutation,
                    baseline_metrics["macro_f1"],
                )
                for permutation in shared_permutations
            ],
            dtype=float,
        )
        importance_rows.append(
            {
                "dataset_id": dataset,
                "representation": "R1_full_native",
                "analysis": "full_block",
                "block": block_name,
                "n_block_features": len(block_features),
                "matched_subset_size": np.nan,
                "importance_mean": float(full_drops.mean()),
                "importance_sd": float(full_drops.std(ddof=1)),
                "interval_low": float(np.quantile(full_drops, 0.025)),
                "interval_high": float(np.quantile(full_drops, 0.975)),
                "importance_per_feature": float(
                    full_drops.mean() / len(block_features)
                ),
                "n_repeats": N_PERM,
                "interval_interpretation": "permutation distribution, not sampling CI",
            }
        )

        if len(block_features) < 2:
            raise RuntimeError("Matched block analysis requires at least two features.")
        matched_drops = []
        for permutation in shared_permutations:
            selected_features = rng.choice(
                block_features, size=2, replace=False
            ).tolist()
            selected_columns = [position[feature] for feature in selected_features]
            matched_drops.append(
                permuted_macro_f1_drop(
                    bundle["model"],
                    X,
                    y_true,
                    bundle["class_order"],
                    selected_columns,
                    permutation,
                    baseline_metrics["macro_f1"],
                )
            )
        matched_drops = np.asarray(matched_drops, dtype=float)
        importance_rows.append(
            {
                "dataset_id": dataset,
                "representation": "R1_full_native",
                "analysis": "matched_two_feature",
                "block": block_name,
                "n_block_features": len(block_features),
                "matched_subset_size": 2,
                "importance_mean": float(matched_drops.mean()),
                "importance_sd": float(matched_drops.std(ddof=1)),
                "interval_low": float(np.quantile(matched_drops, 0.025)),
                "interval_high": float(np.quantile(matched_drops, 0.975)),
                "importance_per_feature": float(matched_drops.mean() / 2.0),
                "n_repeats": N_PERM,
                "interval_interpretation": "permutation distribution, not sampling CI",
            }
        )

    for delta in SCALE_LEVELS:
        perturbed = X.copy()
        factor = 1.0 + delta
        for feature, exponent in SCALE_EXPONENTS[dataset].items():
            if feature not in position:
                raise RuntimeError(
                    f"Scale exponent refers to missing feature: {dataset}/{feature}"
                )
            perturbed[:, position[feature]] *= factor ** exponent
        probabilities = aligned_proba(
            bundle["model"], perturbed, bundle["class_order"]
        )
        output_metrics = metrics(y_true, probabilities, bundle["class_order"])
        output_labels = labels_from_proba(probabilities, bundle["class_order"])
        scale_rows.append(
            {
                "dataset_id": dataset,
                "representation": "R1_full_native",
                "relative_scale_bias": delta,
                "scale_bias_percent": 100.0 * delta,
                "macro_f1": output_metrics["macro_f1"],
                "macro_f1_drop": baseline_metrics["macro_f1"]
                - output_metrics["macro_f1"],
                "accuracy": output_metrics["accuracy"],
                "mcc": output_metrics["mcc"],
                "label_flip_rate": float(np.mean(output_labels != baseline_labels)),
                "interpretation": (
                    "systematic image-space scale transformation; "
                    "not a calibrated physical-size change"
                ),
            }
        )

controlled_importance = pd.DataFrame(importance_rows)
controlled_scale = pd.DataFrame(scale_rows)
controlled_importance.to_csv(
    TABLES / "Table_09_controlled_R1_size_shape_permutation.csv",
    index=False,
    encoding="utf-8-sig",
)
controlled_scale.to_csv(
    TABLES / "Table_10_controlled_R1_scale_bias.csv",
    index=False,
    encoding="utf-8-sig",
)
CONTROLLED_ANALYSES_READY = True
print("Controlled R1 analyses: PASSED")
display(controlled_importance)
display(controlled_scale)

In [ ]:
# ============================================================
# 7. Revised champions, calibration audit, and explicit units
# ============================================================
require_state("CONTROLLED_ANALYSES_READY")
CALIBRATION_STAGE_READY = False

champions = (
    selected.sort_values(
        ["dataset_id", "macro_f1_mean", "log_loss_mean", "representation"],
        ascending=[True, False, True, True],
    )
    .groupby("dataset_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)


def ece(y_true, probabilities, class_order, n_bins=N_BINS):
    predictions = labels_from_proba(probabilities, class_order)
    confidence = probabilities.max(axis=1)
    correct = (predictions == np.asarray(y_true, dtype=str)).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    total = 0.0
    maximum = 0.0
    for bin_index in range(n_bins):
        if bin_index == n_bins - 1:
            mask = (confidence >= edges[bin_index]) & (
                confidence <= edges[bin_index + 1]
            )
        else:
            mask = (confidence >= edges[bin_index]) & (
                confidence < edges[bin_index + 1]
            )
        if not mask.any():
            continue
        accuracy = float(correct[mask].mean())
        mean_confidence = float(confidence[mask].mean())
        gap = abs(accuracy - mean_confidence)
        total += float(mask.mean()) * gap
        maximum = max(maximum, gap)
        rows.append(
            {
                "bin_index": bin_index,
                "lower": edges[bin_index],
                "upper": edges[bin_index + 1],
                "count": int(mask.sum()),
                "accuracy": accuracy,
                "confidence": mean_confidence,
                "gap": gap,
            }
        )
    return float(total), float(maximum), pd.DataFrame(rows)


def certainty(probabilities, name):
    sorted_probabilities = np.sort(probabilities, axis=1)
    if name == "max_probability":
        return sorted_probabilities[:, -1]
    if name == "top2_margin":
        return sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
    raise ValueError(name)


calibration_rows = []
selective_rows = []
for _, champion in champions.iterrows():
    dataset = champion["dataset_id"]
    representation = champion["representation"]
    bundle = bundles[(dataset, representation)]

    for partition in ["calibration", "test"]:
        frame = bundle[partition]
        y_true = frame["class_label"].astype(str).to_numpy()
        probability_sets = {
            "native": (
                bundle["cal_native"]
                if partition == "calibration"
                else bundle["test_native"]
            ),
            "temperature_scaled": (
                bundle["cal_scaled"]
                if partition == "calibration"
                else bundle["test_scaled"]
            ),
        }
        bins_by_status = {}
        for status, probabilities in probability_sets.items():
            output_metrics = metrics(
                y_true, probabilities, bundle["class_order"]
            )
            ece_value, maximum_calibration_error, bins = ece(
                y_true, probabilities, bundle["class_order"]
            )
            calibration_rows.append(
                {
                    "dataset_id": dataset,
                    "representation": representation,
                    "partition": partition,
                    "probability_status": status,
                    "temperature": bundle["temperature"],
                    "log_loss": output_metrics["log_loss"],
                    "brier_multiclass": output_metrics["brier_multiclass"],
                    "ece_equal_width": ece_value,
                    "maximum_calibration_error": maximum_calibration_error,
                }
            )
            bins_path = RESULTS / f"reliability_bins_{dataset}_{partition}_{status}.csv"
            bins.to_csv(bins_path, index=False, encoding="utf-8-sig")
            bins_by_status[status] = bins

        figure, axis = plt.subplots(figsize=(7, 6))
        axis.plot([0.0, 1.0], [0.0, 1.0], "--", label="Perfect calibration")
        axis.plot(
            bins_by_status["native"]["confidence"],
            bins_by_status["native"]["accuracy"],
            marker="o",
            label="Native",
        )
        axis.plot(
            bins_by_status["temperature_scaled"]["confidence"],
            bins_by_status["temperature_scaled"]["accuracy"],
            marker="o",
            label="Temperature-scaled",
        )
        axis.set_xlabel("Mean predicted confidence")
        axis.set_ylabel("Empirical accuracy")
        axis.set_title(f"Reliability diagram — {dataset} — {partition}")
        axis.grid(True, alpha=0.3)
        axis.legend()
        figure.tight_layout()
        figure.savefig(
            FIGURES / f"reliability_{dataset}_{partition}.png",
            dpi=300,
            bbox_inches="tight",
        )
        figure.savefig(
            FIGURES / f"reliability_{dataset}_{partition}.pdf",
            bbox_inches="tight",
        )
        plt.show()

    score_name = SELECTIVE_SCORE[dataset]
    calibration = bundle["calibration"].copy()
    calibration["score"] = certainty(bundle["cal_scaled"], score_name)
    calibration_group = calibration.groupby(
        "duplicate_group_id", as_index=False, sort=True
    ).agg(score=("score", "min"), class_label=("class_label", "first"))
    sorted_scores = np.sort(calibration_group["score"].to_numpy(dtype=float))
    retained_target = math.ceil(TARGET_SELECTIVE_COVERAGE * len(sorted_scores))
    threshold = float(sorted_scores[max(0, len(sorted_scores) - retained_target)])

    test = bundle["test"].copy()
    test["score"] = certainty(bundle["test_scaled"], score_name)
    test["prediction"] = labels_from_proba(
        bundle["test_native"], bundle["class_order"]
    )
    for unit in ["row", "group"]:
        evaluation = (
            test
            if unit == "row"
            else test.groupby("duplicate_group_id", as_index=False, sort=True).agg(
                class_label=("class_label", "first"),
                prediction=("prediction", "first"),
                score=("score", "min"),
            )
        )
        keep = evaluation["score"].to_numpy(dtype=float) >= threshold
        y_true = evaluation["class_label"].astype(str).to_numpy()
        predictions = evaluation["prediction"].astype(str).to_numpy()
        baseline_risk = 1.0 - accuracy_score(y_true, predictions)
        if keep.any():
            selective_accuracy = accuracy_score(y_true[keep], predictions[keep])
            selective_risk = 1.0 - selective_accuracy
        else:
            selective_accuracy = np.nan
            selective_risk = np.nan
        selective_rows.append(
            {
                "dataset_id": dataset,
                "representation": representation,
                "certainty_score": score_name,
                "calibration_unit": "duplicate_group_id",
                "evaluation_unit": unit,
                "target_calibration_coverage": TARGET_SELECTIVE_COVERAGE,
                "certainty_threshold": threshold,
                "n_total": len(evaluation),
                "n_retained": int(keep.sum()),
                "coverage": float(keep.mean()),
                "selective_accuracy": float(selective_accuracy),
                "selective_risk": float(selective_risk),
                "baseline_risk": float(baseline_risk),
                "risk_reduction": float(baseline_risk - selective_risk),
            }
        )

calibration_audit = pd.DataFrame(calibration_rows)
selective = pd.DataFrame(selective_rows)
champions.to_csv(
    TABLES / "Table_11_revised_training_selected_champions.csv",
    index=False,
    encoding="utf-8-sig",
)
calibration_audit.to_csv(
    TABLES / "Table_12_calibration_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
selective.to_csv(
    TABLES / "Table_13_selective_results_row_and_group_units.csv",
    index=False,
    encoding="utf-8-sig",
)
CALIBRATION_STAGE_READY = True
print("Calibration and selective stage: PASSED")
display(champions)
display(calibration_audit)
display(selective)

In [ ]:
# ============================================================
# 8. Conformal empty-set diagnostics on revised champions
# ============================================================
require_state("CALIBRATION_STAGE_READY")
CONFORMAL_STAGE_READY = False


def stable_uniform(seed, dataset, group_id, class_label, method):
    token = f"{seed}|{dataset}|{group_id}|{class_label}|{method}"
    integer = int.from_bytes(
        hashlib.sha256(token.encode("utf-8")).digest()[:8], "big"
    )
    return (integer + 0.5) / (2**64)


def score_matrix(probabilities, class_order, dataset, group_ids, method):
    probabilities = np.asarray(probabilities, dtype=float)
    class_order = np.asarray(class_order, dtype=str)
    output = np.zeros_like(probabilities)
    for row_index, row in enumerate(probabilities):
        ranked = np.argsort(-row, kind="mergesort")
        cumulative_before = 0.0
        for class_index in ranked:
            probability = row[class_index]
            if method == "LAC":
                score = 1.0 - probability
            elif method == "APS_randomized":
                score = cumulative_before + stable_uniform(
                    SEEDS["conformal"],
                    dataset,
                    group_ids[row_index],
                    class_order[class_index],
                    method,
                ) * probability
            elif method == "APS_deterministic":
                # Standard non-randomized cumulative APS score.
                score = cumulative_before + probability
            else:
                raise ValueError(method)
            output[row_index, class_index] = score
            cumulative_before += probability
    return output


def conformal_quantile(values, coverage):
    values = np.sort(np.asarray(values, dtype=float))
    index = min(
        max(math.ceil((len(values) + 1) * coverage), 1), len(values)
    )
    return float(values[index - 1]), index


conformal_rows = []
conformal_class_rows = []
quantile_rows = []
for _, champion in champions.iterrows():
    dataset = champion["dataset_id"]
    representation = champion["representation"]
    bundle = bundles[(dataset, representation)]
    class_order = bundle["class_order"]
    position = {label: index for index, label in enumerate(class_order)}
    calibration = bundle["calibration"].copy()
    test = bundle["test"].copy()
    calibration_true = np.asarray(
        [position[label] for label in calibration["class_label"].astype(str)],
        dtype=int,
    )
    test_true = np.asarray(
        [position[label] for label in test["class_label"].astype(str)],
        dtype=int,
    )

    for method in ["LAC", "APS_randomized", "APS_deterministic"]:
        calibration_scores = score_matrix(
            bundle["cal_scaled"],
            class_order,
            dataset,
            calibration["duplicate_group_id"].astype(str).to_numpy(),
            method,
        )
        test_scores = score_matrix(
            bundle["test_scaled"],
            class_order,
            dataset,
            test["duplicate_group_id"].astype(str).to_numpy(),
            method,
        )
        true_scores = calibration_scores[
            np.arange(len(calibration)), calibration_true
        ]
        group_scores = (
            pd.DataFrame(
                {
                    "group": calibration["duplicate_group_id"].astype(str),
                    "score": true_scores,
                }
            )
            .groupby("group", as_index=False, sort=True)["score"]
            .max()
        )

        for nominal in NOMINAL_COVERAGES:
            quantile, order_index = conformal_quantile(
                group_scores["score"], nominal
            )
            quantile_rows.append(
                {
                    "dataset_id": dataset,
                    "representation": representation,
                    "method": method,
                    "nominal_coverage": nominal,
                    "qhat": quantile,
                    "n_calibration_groups": len(group_scores),
                    "finite_sample_order_index": order_index,
                }
            )
            prediction_sets = test_scores <= quantile
            row_results = pd.DataFrame(
                {
                    "duplicate_group_id": test["duplicate_group_id"].astype(str),
                    "class_label": test["class_label"].astype(str),
                    "covered": prediction_sets[
                        np.arange(len(test)), test_true
                    ],
                    "set_size": prediction_sets.sum(axis=1),
                }
            )
            for unit in ["row", "group"]:
                evaluation = (
                    row_results
                    if unit == "row"
                    else row_results.groupby(
                        "duplicate_group_id", as_index=False, sort=True
                    ).agg(
                        class_label=("class_label", "first"),
                        covered=("covered", "min"),
                        set_size=("set_size", "max"),
                    )
                )
                empty_rate = float(
                    np.mean(evaluation["set_size"].to_numpy(dtype=int) == 0)
                )
                coverage = float(evaluation["covered"].mean())
                conformal_rows.append(
                    {
                        "dataset_id": dataset,
                        "representation": representation,
                        "method": method,
                        "nominal_coverage": nominal,
                        "evaluation_unit": unit,
                        "coverage": coverage,
                        "mean_set_size": float(evaluation["set_size"].mean()),
                        "singleton_rate": float(
                            np.mean(
                                evaluation["set_size"].to_numpy(dtype=int) == 1
                            )
                        ),
                        "empty_set_rate": empty_rate,
                        "upper_bound_from_nonempty_rate": 1.0 - empty_rate,
                        "coverage_margin_below_nonempty_upper_bound": (
                            1.0 - empty_rate - coverage
                        ),
                        "n_evaluation_units": len(evaluation),
                        "empty_sets_counted_as_noncovering": True,
                    }
                )
                for class_label in class_order:
                    subset = evaluation.loc[
                        evaluation["class_label"] == class_label
                    ]
                    conformal_class_rows.append(
                        {
                            "dataset_id": dataset,
                            "representation": representation,
                            "method": method,
                            "nominal_coverage": nominal,
                            "evaluation_unit": unit,
                            "class_label": class_label,
                            "support": len(subset),
                            "coverage": float(subset["covered"].mean()),
                            "mean_set_size": float(subset["set_size"].mean()),
                            "empty_set_rate": float(
                                np.mean(
                                    subset["set_size"].to_numpy(dtype=int) == 0
                                )
                            ),
                        }
                    )

conformal = pd.DataFrame(conformal_rows)
conformal_classwise = pd.DataFrame(conformal_class_rows)
quantiles = pd.DataFrame(quantile_rows)
conformal.to_csv(
    TABLES / "Table_14_revised_conformal_empty_set_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)
conformal_classwise.to_csv(
    TABLES / "Table_15_revised_conformal_classwise.csv",
    index=False,
    encoding="utf-8-sig",
)
quantiles.to_csv(
    TABLES / "Table_16_revised_conformal_quantiles.csv",
    index=False,
    encoding="utf-8-sig",
)
CONFORMAL_STAGE_READY = True
print("Conformal diagnostics: PASSED")
display(conformal)

In [ ]:
# ============================================================
# 9. Quantitative leakage audits
# ============================================================
require_state("CONFORMAL_STAGE_READY")
LEAKAGE_STAGE_READY = False

naive_rows = []
contamination_rows = []
for dataset_index, dataset in enumerate(DATASETS):
    bundle = bundles[(dataset, "R1_full_native")]
    full = frames[(dataset, "R1_full_native")].copy()
    feature_list = bundle["features"]
    selected_r1 = selected.loc[
        (selected["dataset_id"] == dataset)
        & (selected["representation"] == "R1_full_native")
    ].iloc[0]
    parameters = {
        "C": float(selected_r1["C"]),
        "gamma": normalize_gamma(selected_r1["gamma"]),
    }
    official = point.loc[
        (point["dataset_id"] == dataset)
        & (point["representation"] == "R1_full_native")
    ].iloc[0]

    for repeat in range(N_NAIVE):
        seed = SEEDS["naive"] + dataset_index * 1000 + repeat
        development, test = train_test_split(
            full,
            test_size=0.20,
            stratify=full["class_label"],
            random_state=seed,
        )
        train, calibration = train_test_split(
            development,
            test_size=0.25,
            stratify=development["class_label"],
            random_state=seed,
        )
        model = build_model(parameters, seed)
        start = time.perf_counter()
        model.fit(
            train[feature_list].to_numpy(dtype=float),
            train["class_label"].astype(str).to_numpy(),
        )
        elapsed = time.perf_counter() - start
        class_order = np.sort(train["class_label"].astype(str).unique())
        probabilities = aligned_proba(
            model, test[feature_list].to_numpy(dtype=float), class_order
        )
        output_metrics = metrics(
            test["class_label"].astype(str).to_numpy(),
            probabilities,
            class_order,
        )
        train_groups = set(train["duplicate_group_id"].astype(str))
        calibration_groups = set(calibration["duplicate_group_id"].astype(str))
        test_groups = set(test["duplicate_group_id"].astype(str))
        naive_rows.append(
            {
                "dataset_id": dataset,
                "repeat": repeat,
                "seed": seed,
                "fit_seconds": elapsed,
                "naive_test_rows": len(test),
                "train_test_group_overlap_count": len(train_groups & test_groups),
                "development_test_group_overlap_count": len(
                    (train_groups | calibration_groups) & test_groups
                ),
                "naive_macro_f1": output_metrics["macro_f1"],
                "official_grouped_macro_f1": official["macro_f1"],
                "naive_minus_official_macro_f1": output_metrics["macro_f1"]
                - official["macro_f1"],
                "naive_accuracy": output_metrics["accuracy"],
                "official_grouped_accuracy": official["accuracy"],
                "naive_minus_official_accuracy": output_metrics["accuracy"]
                - official["accuracy"],
                "interpretation": (
                    "descriptive row-level split audit; test composition "
                    "differs from the frozen grouped test"
                ),
            }
        )

    frozen_train = full.loc[full["partition"] == "train"].copy()
    frozen_test = full.loc[full["partition"] == "test"].copy()
    duplicated_test_groups = frozen_test.groupby("duplicate_group_id").size()
    duplicated_test_groups = duplicated_test_groups.loc[
        duplicated_test_groups > 1
    ].index.astype(str).tolist()

    if duplicated_test_groups:
        leaked = (
            frozen_test.loc[
                frozen_test["duplicate_group_id"].astype(str).isin(
                    duplicated_test_groups
                )
            ]
            .groupby("duplicate_group_id", as_index=False, sort=True)
            .first()
        )
        contaminated_train = pd.concat(
            [frozen_train, leaked], ignore_index=True
        )
        model = build_model(parameters, SEEDS["naive"] + 9999 + dataset_index)
        model.fit(
            contaminated_train[feature_list].to_numpy(dtype=float),
            contaminated_train["class_label"].astype(str).to_numpy(),
        )
        class_order = np.sort(
            contaminated_train["class_label"].astype(str).unique()
        )
        probabilities = aligned_proba(
            model,
            frozen_test[feature_list].to_numpy(dtype=float),
            class_order,
        )
        output_metrics = metrics(
            frozen_test["class_label"].astype(str).to_numpy(),
            probabilities,
            class_order,
        )
        contamination_rows.append(
            {
                "dataset_id": dataset,
                "duplicated_test_groups_injected": len(duplicated_test_groups),
                "injected_training_rows": len(leaked),
                "contaminated_macro_f1": output_metrics["macro_f1"],
                "official_macro_f1": official["macro_f1"],
                "contaminated_minus_official_macro_f1": (
                    output_metrics["macro_f1"] - official["macro_f1"]
                ),
                "contaminated_accuracy": output_metrics["accuracy"],
                "official_accuracy": official["accuracy"],
                "contaminated_minus_official_accuracy": (
                    output_metrics["accuracy"] - official["accuracy"]
                ),
                "interpretation": (
                    "same frozen test; one exact duplicate from each "
                    "duplicated test group was injected into training"
                ),
            }
        )
    else:
        contamination_rows.append(
            {
                "dataset_id": dataset,
                "duplicated_test_groups_injected": 0,
                "injected_training_rows": 0,
                "contaminated_macro_f1": np.nan,
                "official_macro_f1": official["macro_f1"],
                "contaminated_minus_official_macro_f1": np.nan,
                "contaminated_accuracy": np.nan,
                "official_accuracy": official["accuracy"],
                "contaminated_minus_official_accuracy": np.nan,
                "interpretation": (
                    "not applicable; no duplicated frozen-test groups"
                ),
            }
        )

naive_audit = pd.DataFrame(naive_rows)
contamination_audit = pd.DataFrame(contamination_rows)
naive_audit.to_csv(
    TABLES / "Table_17_repeated_naive_row_split_leakage_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
contamination_audit.to_csv(
    TABLES / "Table_18_fixed_test_duplicate_contamination_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
LEAKAGE_STAGE_READY = True
print("Leakage audits: PASSED")
display(naive_audit)
display(contamination_audit)

In [ ]:
# ============================================================
# 10. Confusion matrices and complete classwise metrics
# ============================================================
require_state("LEAKAGE_STAGE_READY")
CLASSWISE_STAGE_READY = False

classwise_rows = []
for _, champion in champions.iterrows():
    dataset = champion["dataset_id"]
    representation = champion["representation"]
    bundle = bundles[(dataset, representation)]
    y_true = bundle["test"]["class_label"].astype(str).to_numpy()
    predictions = labels_from_proba(
        bundle["test_native"], bundle["class_order"]
    )
    matrix = confusion_matrix(
        y_true, predictions, labels=bundle["class_order"]
    )
    matrix_frame = pd.DataFrame(
        matrix, index=bundle["class_order"], columns=bundle["class_order"]
    )
    matrix_frame.to_csv(
        TABLES / f"Table_19_confusion_matrix_{dataset}.csv",
        encoding="utf-8-sig",
    )
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        predictions,
        labels=bundle["class_order"],
        zero_division=0,
    )
    for class_index, class_label in enumerate(bundle["class_order"]):
        classwise_rows.append(
            {
                "dataset_id": dataset,
                "representation": representation,
                "class_label": class_label,
                "precision": float(precision[class_index]),
                "recall": float(recall[class_index]),
                "f1": float(f1[class_index]),
                "support": int(support[class_index]),
            }
        )

    figure, axis = plt.subplots(figsize=(8, 7))
    image = axis.imshow(matrix, aspect="auto")
    axis.set_xticks(np.arange(len(bundle["class_order"])))
    axis.set_yticks(np.arange(len(bundle["class_order"])))
    axis.set_xticklabels(bundle["class_order"], rotation=45, ha="right")
    axis.set_yticklabels(bundle["class_order"])
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    axis.set_title(f"Frozen-test confusion matrix — {dataset}")
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                str(matrix[row_index, column_index]),
                ha="center",
                va="center",
                fontsize=8,
            )
    figure.colorbar(image, ax=axis, label="Count")
    figure.tight_layout()
    figure.savefig(
        FIGURES / f"confusion_matrix_{dataset}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES / f"confusion_matrix_{dataset}.pdf",
        bbox_inches="tight",
    )
    plt.show()

classwise = pd.DataFrame(classwise_rows)
classwise.to_csv(
    TABLES / "Table_20_complete_classwise_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
CLASSWISE_STAGE_READY = True
print("Classwise metrics: PASSED")
display(classwise)

In [ ]:
# ============================================================
# 11. Evidence map, validation, manifest, and robust export
# ============================================================
require_state("CLASSWISE_STAGE_READY")
RUN_COMPLETE = False

reviewer_evidence = pd.DataFrame(
    [
        {
            "reviewer_item": "M1",
            "status": "computationally addressed",
            "primary_output": "Tables 01 and 05–08",
            "note": "Complete and reduced invariant representations are separated.",
        },
        {
            "reviewer_item": "M2",
            "status": "computationally addressed",
            "primary_output": "Tables 09–10",
            "note": "Both crops are analyzed on R1; matched two-feature permutation controls block size.",
        },
        {
            "reviewer_item": "M3",
            "status": "computationally addressed",
            "primary_output": "Tables 06–07",
            "note": "Metric bootstrap and exact McNemar inference are separate.",
        },
        {
            "reviewer_item": "M4",
            "status": "requires verified literature integration",
            "primary_output": "",
            "note": "The notebook does not fabricate external benchmark values.",
        },
        {
            "reviewer_item": "M5",
            "status": "requires manuscript clarification",
            "primary_output": "Protocol",
            "note": "All measurements are pixel-based image-space descriptors.",
        },
        {
            "reviewer_item": "M6",
            "status": "computationally addressed",
            "primary_output": "Tables 03, 05, and 06",
            "note": "R2 is reported and contrasted.",
        },
        {
            "reviewer_item": "M7",
            "status": "computationally addressed",
            "primary_output": "Table 08",
            "note": "Equivalence margin is fixed at ±0.01 macro-F1.",
        },
        {
            "reviewer_item": "M8",
            "status": "computationally addressed",
            "primary_output": "Table 12 and reliability figures",
            "note": "ECE, MCE, Brier score, and log-loss are reported.",
        },
        {
            "reviewer_item": "M9",
            "status": "computationally addressed",
            "primary_output": "Tables 14–16",
            "note": "Empty sets are non-covering; LAC and deterministic APS are included.",
        },
        {
            "reviewer_item": "M10",
            "status": "computationally addressed",
            "primary_output": "Tables 17–18",
            "note": "Repeated row-split and fixed-test contamination audits are included.",
        },
        {
            "reviewer_item": "M11",
            "status": "computationally addressed",
            "primary_output": "Table 13",
            "note": "Row-level and group-level selective results are separate.",
        },
        {
            "reviewer_item": "M12",
            "status": "computationally addressed",
            "primary_output": "Tables 19–20 and confusion figures",
            "note": "Complete classwise metrics are reported.",
        },
        {
            "reviewer_item": "M13",
            "status": "requires verified literature update",
            "primary_output": "",
            "note": "Recent references must be verified externally.",
        },
        {
            "reviewer_item": "M14",
            "status": "requires repository action",
            "primary_output": "Manifest and output ZIP",
            "note": "A repository DOI cannot be created by this notebook.",
        },
    ]
)
reviewer_evidence.to_csv(
    RESULTS / "02_reviewer_evidence_map.csv",
    index=False,
    encoding="utf-8-sig",
)

protocol = {
    "protocol_name": "Grain_AI_MDPI_reviewer_driven_reanalysis_v2_corrected",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "seeds": SEEDS,
    "n_bootstrap": N_BOOTSTRAP,
    "n_permutation_repeats": N_PERM,
    "n_naive_split_repeats": N_NAIVE,
    "equivalence_margin_macro_f1": EQUIV_MARGIN,
    "software": {
        "python": sys.version.split()[0],
        "scikit_learn": importlib.metadata.version("scikit-learn"),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": importlib.metadata.version("scipy"),
        "joblib": importlib.metadata.version("joblib"),
        "statsmodels": importlib.metadata.version("statsmodels"),
    },
    "measurement_units": {
        "linear": "pixels",
        "area": "squared pixels",
        "shape_ratios": "dimensionless",
        "physical_unit_conversion": None,
    },
    "inference_families": {
        "bootstrap_holm": "within dataset and metric across prespecified contrasts",
        "mcnemar_holm": "within dataset across prespecified contrasts",
    },
    "restrictions": [
        "No test-based hyperparameter selection.",
        "No test-based representation selection.",
        "No physical-unit interpretation of pixel measurements.",
        "No external benchmark values fabricated.",
        "Empty conformal sets are counted as non-covering.",
        "SMOKE and FAST outputs are not manuscript-grade.",
    ],
}
(PROTOCOL / "reviewer_reanalysis_protocol_v2.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8"
)

expected_candidates = sum(len(value) for value in REPS.values()) * len(
    ACTIVE_RBF_GRID
)
checks = {
    "source_validations_passed": bool(
        bool_series(val06["passed"]).all() and bool_series(val08["passed"]).all()
    ),
    "representation_audit_complete": len(representation_audit)
    == sum(len(value) for value in REPS.values()),
    "dry_complete_invariant_has_eight_features": len(
        features[("dry_bean", "R3_complete_invariant")]
    )
    == 8,
    "all_candidate_evaluations_complete": len(selection) == expected_candidates,
    "one_selected_candidate_per_representation": len(selected)
    == sum(len(value) for value in REPS.values()),
    "all_prediction_bundles_created": len(bundles) == len(selected),
    "bootstrap_contrasts_complete": len(contrasts)
    == sum(len(value) for value in CONTRASTS.values()) * len(ALL_METRICS),
    "mcnemar_contrasts_complete": len(mcnemar)
    == sum(len(value) for value in CONTRASTS.values()),
    "controlled_R1_importance_complete": len(controlled_importance)
    == len(DATASETS) * 4,
    "controlled_R1_scale_complete": len(controlled_scale)
    == len(DATASETS) * len(SCALE_LEVELS),
    "calibration_audit_complete": len(calibration_audit) == len(DATASETS) * 4,
    "selective_units_explicit": set(selective["evaluation_unit"])
    == {"row", "group"},
    "conformal_methods_complete": set(conformal["method"])
    == {"LAC", "APS_randomized", "APS_deterministic"},
    "empty_sets_counted_as_noncovering": bool(
        conformal["empty_sets_counted_as_noncovering"].all()
    ),
    "leakage_audit_complete": len(naive_audit) == len(DATASETS) * N_NAIVE,
    "classwise_metrics_complete": classwise["dataset_id"].nunique()
    == len(DATASETS),
    "no_test_based_reselection": True,
}
validation = pd.DataFrame(
    [{"check": check, "passed": bool(passed)} for check, passed in checks.items()]
)
validation.to_csv(
    RESULTS / "03_final_reviewer_reanalysis_validation.csv",
    index=False,
    encoding="utf-8-sig",
)
if not validation["passed"].all():
    display(validation)
    raise RuntimeError("Final validation failed.")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


manifest_rows = []
for path in sorted(OUTROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(path.relative_to(OUTROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(
    RESULTS / "04_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

if IN_COLAB:
    EXPORT_ROOT = DRIVE_ROOT
else:
    EXPORT_ROOT = Path(
        os.environ.get("NB12_EXPORT_ROOT", str(WORK_ROOT / "04_export"))
    )
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

suffix = "" if RUN_MODE == "FULL" else f"_{RUN_MODE}"
DRIVE_OUTPUT = EXPORT_ROOT / f"NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2{suffix}"
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTROOT, DRIVE_OUTPUT)

FINAL_ZIP = EXPORT_ROOT / f"NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2{suffix}.zip"
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()
with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(OUTROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(OUTROOT)))

RUN_COMPLETE = True
print("Final validation: PASSED")
print("Run mode:", RUN_MODE)
print("Output directory:", DRIVE_OUTPUT)
print("Final ZIP:", FINAL_ZIP)
print("ZIP size MB:", round(FINAL_ZIP.stat().st_size / 1e6, 2))
display(validation)
display(reviewer_evidence)

# Post-run decision rules

1. Use only a `FULL` run for the manuscript and response letter.
2. Read `Table_01_representation_audit.csv` before interpreting the corrected contrasts.
3. The central invariance claim survives only if the Dry Bean `R1_full_native − R3_complete_invariant` effect remains practically important and its paired interval excludes zero.
4. Cross-crop conclusions must rely on the controlled R1 analyses in Tables 09–10.
5. McNemar tests paired correctness outcomes; group bootstrap tests metric differences.
6. Rice equivalence may be claimed only when the complete 90% paired interval lies inside ±0.01 and both TOST tails are below 0.05.
7. Empty conformal sets remain non-covering abstentions.
8. Pixel-based measurements must not be described as calibrated millimetres or square millimetres.
9. `FAST` and `SMOKE` outputs are diagnostic only and must not be reported.